# Assignment 4 — Orchestration with LangGraph

**Goal:** rebuild the multi-agent setup in **LangGraph** (graph-based orchestration), with node-level roles and a **conditional branch**:

> If the topic is **finance** -> use **FinanceAgent**; otherwise -> use **GeneralAgent**.

Same `StateGraph` / `TypedDict` style as your `LangGraph.ipynb`.

In [2]:
%pip install langgraph langchain langchain-community langchain-huggingface ddgs huggingface_hub transformers

Note: you may need to restart the kernel to use updated packages.


## Set up the model

The assignment asks for an **instruction-tuned model** (`Mistral-7B-Instruct-v0.2`). We call it through Hugging Face's hosted endpoint — the same pattern as your `LangChain agent.ipynb`. It reads your token from the `HF_TOKEN` environment variable.

> **No HF inference access?** Swap the cell for a local model, exactly like your `LangGraph.ipynb` did:
> ```python
> from langchain_community.llms import HuggingFacePipeline
> from transformers import pipeline
> llm = HuggingFacePipeline(pipeline=pipeline('text2text-generation', model='google/flan-t5-base', max_new_tokens=128))
> ```

In [3]:
# --- LOCAL MODEL: runs offline, no token needed ---
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

llm = HuggingFacePipeline(pipeline=pipeline(
    'text2text-generation', model='google/flan-t5-base', max_new_tokens=256))
print('LLM ready: local flan-t5-base')

# --- To use Mistral instead (needs HF_TOKEN), comment out the 4 lines above and use this: ---
# import os
# from langchain_huggingface import HuggingFaceEndpoint
# llm = HuggingFaceEndpoint(repo_id='mistralai/Mistral-7B-Instruct-v0.2', task='text-generation',
#     temperature=0.1, max_new_tokens=256, repetition_penalty=1.1,
#     stop_sequences=['Observation:'], huggingfacehub_api_token=os.getenv('HF_TOKEN'))

d:\nihal\datacamp\ML practice\ml_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LLM ready: local flan-t5-base


C:\Users\hp\AppData\Local\Temp\ipykernel_16728\3607214097.py:5: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipeline(


In [4]:
import re
from langchain_community.tools.ddg_search.tool import DuckDuckGoSearchRun

# --- Tool 1: web search (DuckDuckGo) ---
_ddg = DuckDuckGoSearchRun()
_ddg.api_wrapper.backend = 'html'  # real page snippets

def search_tool(query: str) -> str:
    """Search the web and return a short text snippet."""
    try:
        return _ddg.run(query)[:500]
    except Exception as e:
        return f'Search error: {e}'

# --- Tool 2: calculator (safe arithmetic only) ---
def calculator_tool(expr: str) -> str:
    """Evaluate a math expression like '4.4 * 0.05'."""
    if not re.fullmatch(r'[0-9\.\+\-\*\/\(\) ]+', expr):
        return 'Calculator error: invalid expression.'
    try:
        return str(eval(expr))
    except Exception as e:
        return f'Calculator error: {e}'

# Registry the agent will pick from
TOOLS = {'search': search_tool, 'calculator': calculator_tool}
print('Tools available:', list(TOOLS))

Tools available: ['search', 'calculator']


## Shared graph state

Every node reads from and writes to this shared `GraphState` — the communication channel between agents.

In [5]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END

class GraphState(TypedDict, total=False):
    query: str
    topic: Optional[str]      # 'finance' or 'general' (set by the router)
    facts: Optional[str]
    report: Optional[str]

## The nodes (agents)

- **router** — classifies the query as `finance` or `general` (this drives the branch).
- **FinanceAgent** — searches, then pulls out figures relevant to finance.
- **GeneralAgent** — searches and answers generally.
- **ReportAgent** — writes the final summary (both branches converge here).

In [6]:
FINANCE_WORDS = ('gdp', 'stock', 'price', 'revenue', 'inflation', 'market', 'economy', 'profit', 'interest rate')

def router(state):
    q = state['query'].lower()
    state['topic'] = 'finance' if any(w in q for w in FINANCE_WORDS) else 'general'
    print(f"[Router] topic = {state['topic']}")
    return state

def finance_agent(state):
    print('[FinanceAgent] handling a finance query')
    facts = search_tool(state['query'])
    nums = re.findall(r'\d+(?:\.\d+)?', facts)
    state['facts'] = facts + (f"  [key figures: {', '.join(nums[:3])}]" if nums else '')
    return state

def general_agent(state):
    print('[GeneralAgent] handling a general query')
    state['facts'] = search_tool(state['query'])
    return state

def report_agent(state):
    print('[ReportAgent] writing summary')
    prompt = f"Summarize for a human in 1-2 sentences.\nQuestion: {state['query']}\nFacts: {state['facts']}"
    state['report'] = llm.invoke(prompt).strip()
    return state

## Build the graph with a conditional edge

`add_conditional_edges` is the branch: after `router`, a small function reads `state['topic']` and sends the flow to either `finance` or `general`. Both then converge on `report`.

In [7]:
workflow = StateGraph(GraphState)
workflow.add_node('router', router)
workflow.add_node('finance', finance_agent)
workflow.add_node('general', general_agent)
workflow.add_node('report', report_agent)

workflow.set_entry_point('router')

# THE CONDITIONAL BRANCH: route based on the topic the router decided
workflow.add_conditional_edges(
    'router',
    lambda state: state['topic'],      # returns 'finance' or 'general'
    {'finance': 'finance', 'general': 'general'},
)

# both branches converge on the report node, then end
workflow.add_edge('finance', 'report')
workflow.add_edge('general', 'report')
workflow.add_edge('report', END)

graph = workflow.compile()
print('Graph compiled.')

ValueError: 'report' is already being used as a state key

## Run both branches

In [ ]:
print('========== FINANCE QUERY ==========')
r1 = graph.invoke({'query': 'What was the GDP of Germany in 2023?'})
print('\nReport:', r1['report'])

print('\n========== GENERAL QUERY ==========')
r2 = graph.invoke({'query': 'Who painted the Mona Lisa?'})
print('\nReport:', r2['report'])

## Visualize the orchestration graph (same networkx style as your LangGraph notebook):

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

g_obj = graph.get_graph()
edges = g_obj.edges if hasattr(g_obj, 'edges') else []
nx_graph = nx.DiGraph()
for e in edges:
    src = getattr(e, 'source', e[0] if isinstance(e, (tuple, list)) else None)
    tgt = getattr(e, 'target', e[1] if isinstance(e, (tuple, list)) else None)
    if src and tgt:
        nx_graph.add_edge(src, tgt)

plt.figure(figsize=(7, 5))
pos = nx.spring_layout(nx_graph, seed=42)
nx.draw_networkx(nx_graph, pos, with_labels=True, node_color='lightgreen',
                 node_size=2600, font_size=10, arrows=True)
plt.title('LangGraph Orchestration (with finance/general branch)')
plt.axis('off'); plt.show()

## Reflection

LangGraph turns the hand-written coordinator from Assignment 3 into an explicit **graph**: nodes are agents, edges are the flow, and `add_conditional_edges` gives a real **decision point**. The `router` classifies the query and the graph routes finance questions to the **FinanceAgent** and everything else to the **GeneralAgent**, then both converge on the **ReportAgent**. Compared to the manual sequential controller, the graph makes the branching logic visible, reusable, and easy to extend (add more nodes or conditions without rewriting the control flow).